# 01. Netflix Data Preprocessing, Database & Recommendation Engine
---
**Dataset:** `netflix_titles.csv` (8,807 records, 12 attributes)  
**Objective:** Complete data engineering pipeline — ingestion, missing data audit, anomaly resolution (shifted durations), feature engineering, SQLite 3NF relational database construction, TF-IDF content soup modeling, and grounded explainable recommendations.


## 1. Environment Setup & Raw Data Ingestion
We import the required libraries and load the raw dataset.

In [ ]:
import os
import re
import json
import time
import sqlite3
import string
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Paths
RAW_CSV = Path("../data/raw/netflix_titles.csv")
if not RAW_CSV.exists():
    RAW_CSV = Path("data/raw/netflix_titles.csv")

PROCESSED_DIR = Path("../data/processed")
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(RAW_CSV, dtype=str)
print(f"Raw Dataset Loaded Successfully: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns.")
raw_df.head(3)

## 2. Preliminary Data Audit & Missing Value Analysis
We check dataset dimensions, duplicate identifiers, and column completeness.

In [ ]:
print("=== Dataset Health Audit ===")
print(f"Total Rows            : {len(raw_df)}")
print(f"Duplicate Rows        : {raw_df.duplicated().sum()}")
print(f"Duplicate Show IDs    : {raw_df['show_id'].duplicated().sum()}")

missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(2)
audit_df = pd.DataFrame({"Missing Values": missing, "Percentage (%)": missing_pct})
print("\n=== Missing Values by Column ===")
audit_df[audit_df["Missing Values"] > 0]

## 3. Data Cleaning & Anomaly Correction

### 3.1 Resolving the Shifted Duration Anomaly
In records `s5542`, `s5795`, and `s5814` (Louis C.K. standup specials), the runtime was mistakenly placed into the `rating` column (`74 min`, `84 min`, `66 min`) while `duration` remained `NaN`.
We inspect and repair these records so duration coverage is 100% complete.

In [ ]:
# Identify shifted duration records
shifted_mask = raw_df["rating"].str.contains(r"\d+\s*min", na=False)
print(f"Shifted duration anomalies detected: {shifted_mask.sum()}")
raw_df[shifted_mask][["show_id", "title", "rating", "duration"]]

In [ ]:
df = raw_df.copy()

# 1. Strip string whitespace
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": None, "nan": None, "NaN": None, "None": None})

# 2. Fix the shifted duration records
df.loc[shifted_mask & df["duration"].isnull(), "duration"] = df.loc[shifted_mask, "rating"]
df.loc[shifted_mask, "rating"] = "Not Rated"
df["rating"] = df["rating"].fillna("Not Rated")

# 3. Parse Movie Runtime (minutes) & TV Runtime (seasons)
df["duration_raw"] = df["duration"].fillna("Unknown")

def parse_minutes(val, content_type):
    if content_type != "Movie" or not val:
        return 0.0
    m = re.search(r"(\d+)\s*min", str(val), re.IGNORECASE)
    return float(m.group(1)) if m else 0.0

def parse_seasons(val, content_type):
    if content_type != "TV Show" or not val:
        return 0
    m = re.search(r"(\d+)\s*Season", str(val), re.IGNORECASE)
    return int(m.group(1)) if m else 0

df["duration_minutes"] = df.apply(lambda r: parse_minutes(r["duration_raw"], r["type"]), axis=1)
df["duration_seasons"] = df.apply(lambda r: parse_seasons(r["duration_raw"], r["type"]), axis=1)
df["duration_value"] = np.where(df["type"] == "Movie", df["duration_minutes"], df["duration_seasons"])

# 4. Parse dates to ISO 8601 (YYYY-MM-DD)
parsed_dates = pd.to_datetime(df["date_added"].str.strip(), errors="coerce")
df["date_added_iso"] = parsed_dates.dt.strftime("%Y-%m-%d")
df["year_added"] = parsed_dates.dt.year
df["month_added"] = parsed_dates.dt.month
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

# 5. Multi-value tokenization into lists
def split_items(v):
    if pd.isna(v) or v is None:
        return []
    return [item.strip() for item in str(v).split(",") if item.strip()]

df["genre_list"] = df["listed_in"].apply(split_items)
df["country_list"] = df["country"].apply(split_items)
df["director_list"] = df["director"].apply(split_items)
df["cast_list"] = df["cast"].apply(split_items)

df["primary_country"] = df["country_list"].apply(lambda lst: lst[0] if lst else "Unknown")
df["primary_genre"] = df["genre_list"].apply(lambda lst: lst[0] if lst else "Unknown")
df["description"] = df["description"].fillna("No description available.")

print("Data Cleaning Complete! Validated 8,807 rows.")
print(f"Movies: {(df['type'] == 'Movie').sum()} | TV Shows: {(df['type'] == 'TV Show').sum()}")
print(f"Average Movie Duration: {df[df['type']=='Movie']['duration_minutes'].mean():.1f} mins")
print(f"Unique Genres: {df['genre_list'].explode().nunique()} | Unique Countries: {df['country_list'].explode().nunique()}")

## 4. SQLite Relational Database Generation (3NF Schema)
We construct a normalized relational database `netflix.db` storing titles, genres, countries, directors, and actors with junction tables for many-to-many relationships.

In [ ]:
db_path = PROCESSED_DIR / "netflix.db"
conn = sqlite3.connect(str(db_path))
cur = conn.cursor()

# Create normalized tables
cur.executescript('''
DROP TABLE IF EXISTS content_genres;
DROP TABLE IF EXISTS content_countries;
DROP TABLE IF EXISTS content_directors;
DROP TABLE IF EXISTS content_cast;
DROP TABLE IF EXISTS content;
DROP TABLE IF EXISTS genres;
DROP TABLE IF EXISTS countries;
DROP TABLE IF EXISTS directors;
DROP TABLE IF EXISTS actors;

CREATE TABLE content (
    show_id TEXT PRIMARY KEY,
    type TEXT NOT NULL,
    title TEXT NOT NULL,
    date_added TEXT,
    release_year INTEGER,
    rating TEXT,
    duration TEXT,
    duration_minutes REAL,
    duration_seasons INTEGER,
    description TEXT
);

CREATE TABLE genres (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL);
CREATE TABLE countries (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL);
CREATE TABLE directors (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL);
CREATE TABLE actors (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL);

CREATE TABLE content_genres (show_id TEXT, genre_id INTEGER, FOREIGN KEY(show_id) REFERENCES content(show_id), FOREIGN KEY(genre_id) REFERENCES genres(id));
CREATE TABLE content_countries (show_id TEXT, country_id INTEGER, FOREIGN KEY(show_id) REFERENCES content(show_id), FOREIGN KEY(country_id) REFERENCES countries(id));
CREATE TABLE content_directors (show_id TEXT, director_id INTEGER, FOREIGN KEY(show_id) REFERENCES content(show_id), FOREIGN KEY(director_id) REFERENCES directors(id));
CREATE TABLE content_cast (show_id TEXT, actor_id INTEGER, FOREIGN KEY(show_id) REFERENCES content(show_id), FOREIGN KEY(actor_id) REFERENCES actors(id));
''')

# Insert content records
content_rows = []
for _, r in df.iterrows():
    content_rows.append((
        r['show_id'], r['type'], r['title'], r['date_added_iso'],
        int(r['release_year']) if pd.notna(r['release_year']) else None,
        r['rating'], r['duration_raw'], r['duration_minutes'],
        r['duration_seasons'], r['description']
    ))

cur.executemany('''
INSERT INTO content (show_id, type, title, date_added, release_year, rating, duration, duration_minutes, duration_seasons, description)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', content_rows)

# Helper for entity & association tables
def populate_entity(column_name, entity_table, junction_table, fk_name):
    items = sorted({x for lst in df[column_name] for x in lst if x})
    cur.executemany(f'INSERT INTO {entity_table} (name) VALUES (?)', [(i,) for i in items])
    cur.execute(f'SELECT id, name FROM {entity_table}')
    name_to_id = {name: item_id for item_id, name in cur.fetchall()}
    
    links = []
    for _, r in df.iterrows():
        for name in r[column_name]:
            if name in name_to_id:
                links.append((r['show_id'], name_to_id[name]))
    cur.executemany(f'INSERT INTO {junction_table} (show_id, {fk_name}) VALUES (?, ?)', links)

populate_entity('genre_list', 'genres', 'content_genres', 'genre_id')
populate_entity('country_list', 'countries', 'content_countries', 'country_id')
populate_entity('director_list', 'directors', 'content_directors', 'director_id')
populate_entity('cast_list', 'actors', 'content_cast', 'actor_id')

conn.commit()

# Verification query
cur.execute('SELECT COUNT(*) FROM content')
print(f"Database Built & Reconciled: {cur.fetchone()[0]} titles loaded into SQLite 3NF Schema!")
conn.close()

## 5. Machine Learning Recommender Engine

### 5.1 Constructing the Weighted Content Soup
To capture thematic similarity, we weight features:
- **Genres (2x):** Heavily weighted so comedy recommends comedy, horror recommends horror.
- **Plot Synopsis (1x):** Uncovers keyword alignment (e.g. 'alien', 'undercover cop').
- **Director & Lead Cast (1x):** Matches cinematic style and actor preferences.
- **Country & Type (1x):** Preserves cultural context and format.

In [ ]:
def build_content_soup(row):
    parts = []
    parts.append(str(row.get('description', '')))
    genres = " ".join(row.get('genre_list', []))
    parts.append(genres)
    parts.append(genres)  # 2x weight
    if row.get('director') and pd.notna(row.get('director')):
        parts.append(str(row['director']))
    cast = row.get('cast_list', [])[:5]
    parts.append(" ".join(cast))
    parts.append(" ".join(row.get('country_list', [])))
    parts.append(str(row.get('rating', '')))
    parts.append(str(row.get('type', '')))
    
    clean_text = " ".join(parts).lower()
    clean_text = clean_text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", clean_text).strip()

df['content_soup'] = df.apply(build_content_soup, axis=1)
print("Sample Content Soup (First Row):")
print(df['content_soup'].iloc[0][:260], "...")

### 5.2 TF-IDF Vectorization
We transform the text soup into a sparse TF-IDF matrix using 10,000 unigram/bigram features and sublinear term frequency.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True
)
tfidf_matrix = tfidf.fit_transform(df['content_soup'])
print(f"TF-IDF Feature Matrix Shape: {tfidf_matrix.shape}")
print(f"Matrix Memory Size        : {tfidf_matrix.data.nbytes / (1024*1024):.2f} MB (Sparse CSR Format)")

### 5.3 On-the-Fly Cosine Similarity Calculation (~4 ms)
Instead of saving a 310 MB dense similarity matrix, we compute similarity for any title on-the-fly using the dot product in just ~4 milliseconds.

In [ ]:
# Benchmark on 'Stranger Things'
query_title = "Stranger Things"
query_row = df[df["title"].str.lower() == query_title.lower()].iloc[0]
query_idx = df.index[df["title"].str.lower() == query_title.lower()][0]

t0 = time.time()
query_vec = tfidf_matrix[query_idx:query_idx+1]
scores = linear_kernel(query_vec, tfidf_matrix).flatten()
elapsed_ms = (time.time() - t0) * 1000

print(f"Calculated Cosine Similarity across all 8,807 titles in: {elapsed_ms:.2f} ms")

# Top 5 similar titles (excluding self)
score_pairs = [(idx, score) for idx, score in enumerate(scores) if idx != query_idx]
score_pairs.sort(key=lambda x: x[1], reverse=True)
top_5 = score_pairs[:5]

### 5.4 Grounded Recommendation Explainability
For every recommendation, we inspect metadata intersections with the query title to explain *why* it was recommended (preventing false or generic recommendations).

In [ ]:
q_genres = set(query_row["genre_list"])
q_countries = set(query_row["country_list"])
q_type = query_row["type"]
q_director = query_row["director"]

print(f"=== Top Recommendations for '{query_row['title']}' ({query_row['type']}, {query_row['release_year']}) ===\n")

for rank, (cand_idx, score) in enumerate(top_5, 1):
    cand = df.iloc[cand_idx]
    shared_genres = q_genres & set(cand["genre_list"])
    shared_countries = q_countries & set(cand["country_list"])
    same_format = (q_type == cand["type"])
    same_dir = (pd.notna(q_director) and q_director == cand["director"])
    pct = int(round(score * 100))
    
    print(f"{rank}. {cand['title']} ({cand['type']}, {cand['release_year']}) — {pct}% Match")
    if shared_genres:
        print(f"   ✓ Shared Genres: {', '.join(shared_genres)}")
    if shared_countries:
        print(f"   ✓ Shared Country: {', '.join(shared_countries)}")
    if same_format:
        print(f"   ✓ Same Format: {cand['type']}")
    if same_dir:
        print(f"   ✓ Same Director: {cand['director']}")
    print(f"   Synopsis: {cand['description'][:100]}...")
    print()

## 6. Saving Final Cleaned Artifacts
We export the processed DataFrame and TF-IDF model for use by the web application.

In [ ]:
import pickle

# Save cleaned DataFrame
df.to_pickle(str(PROCESSED_DIR / "netflix_cleaned.pkl"))

# Save TF-IDF model bundle
with open(PROCESSED_DIR / "recommender_tfidf.pkl", "wb") as f:
    pickle.dump({"vectorizer": tfidf, "matrix": tfidf_matrix}, f)

print("All artifacts saved successfully to data/processed/")
print("- netflix_cleaned.pkl")
print("- netflix.db")
print("- recommender_tfidf.pkl")